In [1]:
%pip install coverage

Note: you may need to restart the kernel to use updated packages.


In [5]:
import multiconductor as mc
import multiconductor.shortcircuit as sc
from multiconductor.pycci.std_types import create_std_types


def _build_minimal_mc_net():
    net = mc.create_empty_network(sn_mva=1)
    create_std_types(
        net,
        {
            "lv_line": {
                "r_ohm_per_km": 0.1,
                "x_ohm_per_km": 0.2,
                "r0_ohm_per_km": 0.05,
                "x0_ohm_per_km": 0.1,
                "c_nf_per_km": 100,
                "c0_nf_per_km": 50,
                "max_i_ka": 0.2,
            }
        },
        "sequence",
    )

    b0 = mc.create_bus(net, 12.47)
    b1 = mc.create_bus(net, 0.48)

    mc.create_ext_grid(
        net,
        bus=b0,
        from_phase=range(1, 4),
        to_phase=0,
        vm_pu=1.0,
        va_degree=0.0,
        r_ohm=0.01,
        x_ohm=0.1,
        name="slack",
    )
    net.ext_grid["s_sc_max_mva"] = 1000.0
    net.ext_grid["rx_max"] = 0.1
    net.ext_grid["x0x_max"] = 1.0
    net.ext_grid["r0x0_max"] = 0.1
    mc.create_line(
        net,
        std_type="lv_line",
        model_type="sequence",
        from_bus=b0,
        from_phase=range(1, 4),
        to_bus=b1,
        to_phase=range(1, 4),
        length_km=0.1,
        name="line1",
    )
    mc.create_asymmetric_load(
        net,
        b1,
        from_phase=range(1, 4),
        to_phase=0,
        p_mw=(0.01, 0.01, 0.01),
        q_mvar=(0.005, 0.005, 0.005),
    )
    return net



net = _build_minimal_mc_net()
mc.run_pf(net, tol_vmag_pu=1e-9, tol_vang_rad=1e-9)
print("res_bus" in net and not net.res_bus.empty)

sc.calc_sc(net, case="max", fault="LLL", branch_results=False)
print("res_bus_sc" in net and not net.res_bus_sc.empty)


net.res_bus_sc


True
True


ikss_ka  skss_mw  rk_ohm  xk_ohm
index phase                                  
0     0        46.30  1000.00    0.02    0.17
      1        46.30  1000.00    0.02    0.17
      2        46.30  1000.00    0.02    0.17
      3        46.30  1000.00    0.02    0.17
1     0      1070.95   890.37    0.00    0.00
      1      1070.95   890.37    0.00    0.00
      2      1070.95   890.37    0.00    0.00
      3      1070.95   890.37    0.00    0.00

In [1]:
import pickle

with open("/Users/admin/git/ppmc4sce/backend/ST_CHARLES.pkl", "rb") as f:
    net = pickle.load(f)

net    

This pandapower network includes the following parameter tables:
   - bus (1960 elements)
   - ext_grid_sequence (3 elements)
   - asymmetric_load (183 elements)
   - asymmetric_sgen (166 elements)
   - line (369 elements)
   - switch (782 elements)
   - trafo1ph (366 elements)
   - controller (19 elements)
   - asymmetric_shunt (15 elements)

In [2]:
import multiconductor as mc
import multiconductor.shortcircuit as sc

mc.run_pf(net, tol_vmag_pu=1e-9, tol_vang_rad=1e-9)
net.res_bus

vm_pu   va_degree          p_mw        q_mvar  \
index phase                                                     
0     0           NaN         NaN -0.000000e+00 -0.000000e+00   
      1      1.011769   -0.793538  1.172068e-02  6.173128e-03   
      2      1.011890 -120.779244  1.172054e-02  6.176793e-03   
      3      1.012048  119.207665  1.172378e-02  6.175079e-03   
1     0           NaN         NaN -0.000000e+00 -0.000000e+00   
...               ...         ...           ...           ...   
488   3      1.002406  119.178697  1.751257e-02  9.226479e-03   
489   0      0.000000    0.000000 -0.000000e+00 -0.000000e+00   
      1      1.036997   -0.346648 -1.912505e-13 -8.195374e-14   
      2      1.037061 -120.325935  4.872339e-13  7.495535e-14   
      3      1.037369  119.658687  3.544952e-13  1.693494e-13   

             imbalance_percent  
index phase                     
0     0               0.014403  
      1               0.014403  
      2               0.014403  
      3               0.014403  
1     0               0.014667  
...                        ...  
488   3               0.006859  
489   0               0.021859  
      1               0.021859  
      2               0.021859  
      3               0.021859  

[1960 rows x 5 columns]

In [3]:
net.ext_grid["s_sc_max_mva"] = 1000.0
net.ext_grid["rx_max"] = 0.1
net.ext_grid["x0x_max"] = 1.0
net.ext_grid["r0x0_max"] = 0.1

print("bus rows:", len(net.bus))
print("ext_grid rows:", len(net.ext_grid))
print("ext_grid_sequence rows:", len(net.ext_grid_sequence))
print("res_bus_sc shape:", getattr(net, "res_bus_sc", None).shape if hasattr(net, "res_bus_sc") else None)
print("ext_grid columns:", net.ext_grid.columns.tolist())



sc.calc_sc(net, case="min", fault="1ph", branch_results=False)
net.res_bus_sc

bus rows: 1960
ext_grid rows: 0
ext_grid_sequence rows: 3
res_bus_sc shape: None
ext_grid columns: ['name', 'bus', 'from_phase', 'to_phase', 'vm_pu', 'va_degree', 'r_ohm', 'x_ohm', 'in_service', 's_sc_max_mva', 'rx_max', 'x0x_max', 'r0x0_max']


ikss_ka      skss_mw  rk0_ohm  xk0_ohm  rk1_ohm  xk1_ohm  \
index phase                                                                
0     0            0.00         0.00      NaN      NaN      NaN      NaN   
      1            0.00         0.00      NaN      NaN      NaN      NaN   
      2            0.00         0.00      NaN      NaN      NaN      NaN   
      3            0.00         0.00      NaN      NaN      NaN      NaN   
1     0      2476077.97  22873028.47      0.0      0.0      0.0      0.0   
...                 ...          ...      ...      ...      ...      ...   
488   3            0.00         0.00      NaN      NaN      NaN      NaN   
489   0      4883732.30  45113986.50      0.0      0.0      0.0      0.0   
      1      4883732.30  45113986.50      0.0      0.0      0.0      0.0   
      2      4883732.30  45113986.50      0.0      0.0      0.0      0.0   
      3      4883732.30  45113986.50      0.0      0.0      0.0      0.0   

             rk2_ohm  xk2_ohm  
index phase                    
0     0          NaN      NaN  
      1          NaN      NaN  
      2          NaN      NaN  
      3          NaN      NaN  
1     0          0.0      0.0  
...              ...      ...  
488   3          NaN      NaN  
489   0          0.0      0.0  
      1          0.0      0.0  
      2          0.0      0.0  
      3          0.0      0.0  

[1960 rows x 8 columns]

In [4]:

sc.calc_sc(net)
net.res_bus_sc

ikss_ka       skss_mw  rk_ohm  xk_ohm
index phase                                          
0     0            0.00  0.000000e+00     NaN     NaN
      1            0.00  0.000000e+00     NaN     NaN
      2            0.00  0.000000e+00     NaN     NaN
      3            0.00  0.000000e+00     NaN     NaN
1     0      1858142.59  5.149436e+07     0.0     0.0
...                 ...           ...     ...     ...
488   3            0.00  0.000000e+00     NaN     NaN
489   0      3734185.85  1.034848e+08     0.0     0.0
      1      3734185.85  1.034848e+08     0.0     0.0
      2      3734185.85  1.034848e+08     0.0     0.0
      3      3734185.85  1.034848e+08     0.0     0.0

[1960 rows x 4 columns]

In [5]:
from multiconductor.pycci.cci_ica import run_ica_iterative, run_ica_streamlined

ica_df = run_ica_streamlined(net)
ica_df

,element_type,element_index,from_node,to_node,ica_kw,ica_thermal_kw,ica_ssv_kw,ica_volt_var_kw,ica_protection_kw,ica_reverse_kw,ica_feeder_limit_kw,min_ampacity_A,strat_device,bus_vm_pu,bus_vn_kv
0,line,68,43,466,10568.52,10568.52,0.0,0.0,10568.52,10568.52,10568.52,651.0,source,1.03,16.0
1,line,96,115,347,10568.52,10568.52,0.0,0.0,10568.52,10568.52,10568.52,530.0,source,1.03,16.0
2,line,32,402,58,10568.52,10568.52,0.0,0.0,10568.52,10568.52,10568.52,651.0,source,1.03,16.0
3,line,69,162,77,10568.52,10568.52,0.0,0.0,10568.52,10568.52,10568.52,315.0,source,1.03,16.0
4,line,7,77,13,8980.76,8980.76,0.0,0.0,10568.52,10568.52,10568.52,315.0,source,1.03,16.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
127,line,94,256,108,8729.54,8729.54,0.0,0.0,10568.52,10568.52,10568.52,315.0,source,1.04,16.0
128,line,53,204,79,7759.59,7759.59,0.0,0.0,10568.52,10568.52,10568.52,280.0,source,1.04,16.0
129,line,109,63,419,7759.61,7759.61,0.0,0.0,10568.52,10568.52,10568.52,280.0,source,1.00,16.0
130,line,95,177,66,8729.54,8729.54,0.0,0.0,10568.52,10568.52,10568.52,315.0,source,1.04,16.0


In [6]:
ica2_df = run_ica_iterative(net)
ica2_df

ERROR:multiconductor.pycci.cci_ica:Initialization failed: Factor is exactly singular


In [ ]:
%pip install runcell

In [ ]:
#import pandas as pd
#import pygwalker as pyg

#walker = pyg.walk(ica_df)

Box(children=(HTML(value='\n<div id="ifr-pyg-00064b6079907bd0djURoSxKL0rsCPGa" style="height: auto">\n    <hea…